In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [2]:
comp_df = pd.read_csv("pokemon_competitive_analysis.csv")

print("Shape:", comp_df.shape)
comp_df.head()

Shape: (1303, 23)


,index,name,type1,type2,ability1,ability2,hidden_ability,hp,attack,defense,sp_atk,sp_def,speed,total_stats,legendary,mythical,generation,Smogon_VGC_Usage_2022,Smogon_VGC_Usage_2023,Smogon_VGC_Usage_2024,Worlds_VGC_Usage_2022,Worlds_VGC_Usage_2023,Worlds_VGC_Usage_2024
0,1,bulbasaur,grass,poison,overgrow,No_ability,chlorophyll,45,49,49,65,65,45,318,False,False,generation-i,0.0,NoUsage,0.0,NoUsage,NoUsage,NoUsage
1,2,ivysaur,grass,poison,overgrow,No_ability,chlorophyll,60,62,63,80,80,60,405,False,False,generation-i,0.0,NoUsage,0.0,NoUsage,NoUsage,NoUsage
2,3,venusaur,grass,poison,overgrow,No_ability,chlorophyll,80,82,83,100,100,80,525,False,False,generation-i,20.83915,NoUsage,0.4441,19.62,NoUsage,0.09
3,3,venusaur-mega,grass,poison,thick-fat,No_ability,None,80,100,123,122,120,80,625,False,False,generation-i,NoUsage,NoUsage,NoUsage,NoUsage,NoUsage,NoUsage
4,3,venusaur-gmax,grass,poison,overgrow,No_ability,chlorophyll,80,82,83,100,100,80,525,False,False,generation-i,NoUsage,NoUsage,NoUsage,NoUsage,NoUsage,NoUsage


In [3]:
comp_df.columns = (
    comp_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [4]:
comp_df.info()

print("\nMissing values:")
comp_df.isna().sum().sort_values(ascending=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   index                  1303 non-null   int64 
 1   name                   1303 non-null   object
 2   type1                  1303 non-null   object
 3   type2                  1303 non-null   object
 4   ability1               1303 non-null   object
 5   ability2               1303 non-null   object
 6   hidden_ability         1303 non-null   object
 7   hp                     1303 non-null   int64 
 8   attack                 1303 non-null   int64 
 9   defense                1303 non-null   int64 
 10  sp_atk                 1303 non-null   int64 
 11  sp_def                 1303 non-null   int64 
 12  speed                  1303 non-null   int64 
 13  total_stats            1303 non-null   int64 
 14  legendary              1303 non-null   bool  
 15  mythical             

index                    0
speed                    0
worlds_vgc_usage_2023    0
worlds_vgc_usage_2022    0
smogon_vgc_usage_2024    0
smogon_vgc_usage_2023    0
smogon_vgc_usage_2022    0
generation               0
mythical                 0
legendary                0
total_stats              0
sp_def                   0
name                     0
sp_atk                   0
defense                  0
attack                   0
hp                       0
hidden_ability           0
ability2                 0
ability1                 0
type2                    0
type1                    0
worlds_vgc_usage_2024    0
dtype: int64

In [5]:
print(comp_df.columns.tolist())

['index', 'name', 'type1', 'type2', 'ability1', 'ability2', 'hidden_ability', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed', 'total_stats', 'legendary', 'mythical', 'generation', 'smogon_vgc_usage_2022', 'smogon_vgc_usage_2023', 'smogon_vgc_usage_2024', 'worlds_vgc_usage_2022', 'worlds_vgc_usage_2023', 'worlds_vgc_usage_2024']


In [6]:
raw_usage_cols = [
    "smogon_vgc_usage_2022", "smogon_vgc_usage_2023", "smogon_vgc_usage_2024",
    "worlds_vgc_usage_2022", "worlds_vgc_usage_2023", "worlds_vgc_usage_2024",
]

for col in raw_usage_cols:
    comp_df[col] = (
        comp_df[col]
        .astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    comp_df[col] = pd.to_numeric(comp_df[col], errors="coerce")

comp_df[raw_usage_cols] = comp_df[raw_usage_cols].fillna(0)

In [7]:
comp_df["name"] = comp_df["name"].str.strip().str.lower()

In [8]:
for col in ["type1", "type2", "ability1", "ability2", "hidden_ability"]:
    if col in comp_df.columns:
        comp_df[col] = comp_df[col].astype(str).str.strip().str.lower()

comp_df["type2"] = comp_df["type2"].fillna("none")
comp_df["ability2"] = comp_df["ability2"].replace("no_ability", "none").fillna("none")
comp_df["hidden_ability"] = comp_df["hidden_ability"].fillna("none")

In [9]:
smogon_cols = ["smogon_vgc_usage_2022", "smogon_vgc_usage_2023", "smogon_vgc_usage_2024"]
worlds_cols = ["worlds_vgc_usage_2022", "worlds_vgc_usage_2023", "worlds_vgc_usage_2024"]

comp_df["avg_smogon_usage"] = comp_df[smogon_cols].mean(axis=1)
comp_df["avg_worlds_usage"] = comp_df[worlds_cols].mean(axis=1)
comp_df["avg_total_usage"] = comp_df[
    ["avg_smogon_usage", "avg_worlds_usage"]
].mean(axis=1)

In [10]:
threshold = comp_df["avg_total_usage"].quantile(0.75)

comp_df["high_usage"] = (comp_df["avg_total_usage"] > threshold).astype(int)

comp_df["high_usage"].value_counts()

high_usage
0    977
1    326
Name: count, dtype: int64

In [11]:
full_df = pd.read_csv("pokemon_list.csv")

print("Shape:", full_df.shape)
full_df.head()

Shape: (1072, 13)


,number,name,type1,type2,total,hp,attack,defense,sp_attack,sp_defense,speed,generation,legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,Mega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,3,Gigantamax Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False


In [12]:
full_df.info()

print("\nMissing values:")
full_df.isna().sum().sort_values(ascending=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1072 entries, 0 to 1071
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   number      1072 non-null   int64 
 1   name        1072 non-null   object
 2   type1       1072 non-null   object
 3   type2       574 non-null    object
 4   total       1072 non-null   int64 
 5   hp          1072 non-null   int64 
 6   attack      1072 non-null   int64 
 7   defense     1072 non-null   int64 
 8   sp_attack   1072 non-null   int64 
 9   sp_defense  1072 non-null   int64 
 10  speed       1072 non-null   int64 
 11  generation  1072 non-null   int64 
 12  legendary   1072 non-null   bool  
dtypes: bool(1), int64(9), object(3)
memory usage: 101.7+ KB

Missing values:


type2         498
number          0
name            0
type1           0
total           0
hp              0
attack          0
defense         0
sp_attack       0
sp_defense      0
speed           0
generation      0
legendary       0
dtype: int64

In [13]:
full_df.columns = (
    full_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

full_df.columns

Index(['number', 'name', 'type1', 'type2', 'total', 'hp', 'attack', 'defense',
       'sp_attack', 'sp_defense', 'speed', 'generation', 'legendary'],
      dtype='object')

In [14]:
full_df["name"] = full_df["name"].str.strip().str.lower()

In [15]:
full_df["type2"] = full_df["type2"].fillna("none")

for col in ["type1", "type2"]:
    if col in full_df.columns:
        full_df[col] = full_df[col].astype(str).str.strip().str.lower()

In [16]:
print(full_df.columns)

Index(['number', 'name', 'type1', 'type2', 'total', 'hp', 'attack', 'defense',
       'sp_attack', 'sp_defense', 'speed', 'generation', 'legendary'],
      dtype='object')


In [17]:
stat_cols = ["hp", "attack", "defense", "sp_attack", "sp_defense", "speed"]

for col in stat_cols:
    full_df[col] = pd.to_numeric(full_df[col], errors="coerce")

full_df = full_df.dropna(subset=stat_cols)

In [18]:
full_df = full_df.rename(columns={
    "sp_attack": "sp_atk",
    "sp_defense": "sp_def",
    "total": "total_stats"
})

### Column Alignment

Column names were standardized across datasets (e.g., `sp_attack` → `sp_atk`) to ensure consistency for merging and modeling.

In [19]:
set_comp = set(comp_df["name"])
set_full = set(full_df["name"])

print("In competitive dataset:", len(set_comp))
print("In full dataset:", len(set_full))

print("Only in full dataset:", len(set_full - set_comp))
print("Only in competitive dataset:", len(set_comp - set_full))

In competitive dataset: 1303
In full dataset: 1072
Only in full dataset: 215
Only in competitive dataset: 446


In [20]:
feature_cols = [
    "hp", "attack", "defense",
    "sp_atk", "sp_def", "speed",
    "total_stats"
]

comp_df = comp_df.dropna(subset=feature_cols)
full_df = full_df.dropna(subset=feature_cols)

In [21]:
print("Competitive dataset:", comp_df.shape)
print("Full dataset:", full_df.shape)

comp_df.head()
full_df.head()

Competitive dataset: (1303, 27)
Full dataset: (1072, 13)


,number,name,type1,type2,total_stats,hp,attack,defense,sp_atk,sp_def,speed,generation,legendary
0,1,bulbasaur,grass,poison,318,45,49,49,65,65,45,1,False
1,2,ivysaur,grass,poison,405,60,62,63,80,80,60,1,False
2,3,venusaur,grass,poison,525,80,82,83,100,100,80,1,False
3,3,mega venusaur,grass,poison,625,80,100,123,122,120,80,1,False
4,3,gigantamax venusaur,grass,poison,525,80,82,83,100,100,80,1,False


In [22]:
comp_df.to_csv("pokemon_competitive_clean.csv", index=False)
full_df.to_csv("pokemon_full_clean.csv", index=False)